In [118]:
import torch
import torch.nn as nn
from torchaudio.models import Conformer

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=512, out_channels=256, patch_size=8):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=patch_size, stride=patch_size)
        self.ln = nn.LayerNorm(out_channels)
        self.act = nn.SiLU()

    def forward(self, x):
        # x: (B, T, 512)
        x = x.transpose(1, 2)  # -> (B, 512, T) for Conv1d
        x = self.conv(x)       # -> (B, 256, T_patched)
        x = x.transpose(1, 2)  # -> (B, T_patched, 256) back to BTC
        return self.act(self.ln(x))

class NeuralPatchConformer(nn.Module):
    def __init__(self, num_channels=512, d_model=256, num_classes=40, patch_size=8):
        super().__init__()
        self.patch_size = patch_size
        self.patch_embed = PatchEmbedding(num_channels, d_model, patch_size)
        
        # torchaudio Conformer natively uses (B, T, C)
        self.conformer = Conformer(
            input_dim=d_model,
            num_heads=8,
            ffn_dim=1024,
            num_layers=12,
            depthwise_conv_kernel_size=31
        )
        
        # Classifier output: (B, T, num_classes + 1) for CTC blank
        self.classifier = nn.Linear(d_model, num_classes + 1)

    def forward(self, x, lengths):
        # 1. Patching: (B, T, 512) -> (B, T/8, 256)
        x = self.patch_embed(x)
        
        # 2. Update lengths to match patched time dimension
        patched_lens = torch.div(lengths, self.patch_size, rounding_mode='floor')
        
        # 3. Conformer pass: (B, T_patched, 256)
        x, _ = self.conformer(x, patched_lens)
        
        # 4. Logits
        logits = self.classifier(x)
        return torch.nn.functional.log_softmax(logits, dim=-1), patched_lens

In [119]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class NeuralPhonemeDataset(Dataset):
    def __init__(self, neural_data, phoneme_targets, transform=None):
        """
        neural_data: List of arrays/tensors (shape: 512, Time)
        phoneme_targets: List of lists (integer phoneme IDs)
        """
        self.neural_data = neural_data
        self.phoneme_targets = phoneme_targets
        self.transform = transform

    def __len__(self):
        return len(self.neural_data)

    def __getitem__(self, idx):
        x = torch.tensor(self.neural_data[idx], dtype=torch.float32)
        y = torch.tensor(self.phoneme_targets[idx], dtype=torch.long)
        
        if self.transform:
            x = self.transform(x)
        
        # print(f"__getitem__ idx={idx}: x shape={x.shape}, y shape={y.shape}")
            
        return x, y, x.shape[0], len(y) # data, target, input_len, target_len

In [120]:
class ChannelMask(object):
    def __init__(self, mask_ratio=0.1):
        self.mask_ratio = mask_ratio

    def __call__(self, x):
        # x shape: (512, Time)
        num_channels = x.shape[0]
        mask_count = int(num_channels * self.mask_ratio)
        
        # Randomly select channel indices to zero out
        mask_idx = torch.randperm(num_channels)[:mask_count]
        x[mask_idx, :] = 0
        return x

In [121]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

def collate_fn_brain(batch):
    # Sort batch by length (optional but helpful for some RNN optimizations)
    batch.sort(key=lambda x: x[0].shape[0], reverse=True)
    
    # inputs: list of (T, 512) | targets: list of (L,)
    inputs, targets, input_lens, target_lens = zip(*batch)

    # Padding inputs: Result shape (Batch, Max_Time, 512)
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0.0)
    
    # Padding targets: Result shape (Batch, Max_Target_Len)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)

    return (
        inputs_padded, 
        targets_padded, 
        torch.tensor(input_lens), 
        torch.tensor(target_lens)
    )

In [122]:
from datasets import load_from_disk

train_dataset_raw = load_from_disk('../data/train_dataset')

Loading dataset from disk:   0%|          | 0/58 [00:00<?, ?it/s]

In [123]:
def clean_phonemes(batch):

    results = {"seq_class_ids_clean": []}
    
    for phoneme_ids in batch["seq_class_ids"]:
        # remove 0 entries at the end
        phoneme_ids_clean = [pid for pid in phoneme_ids if pid != 0]
        results["seq_class_ids_clean"].append(phoneme_ids_clean)
    
    return results

processed_dataset = train_dataset_raw.map(
    clean_phonemes, 
    batched=True, 
    remove_columns=train_dataset_raw.column_names
)

In [124]:
# 1. Initialize Augmentation
transform = ChannelMask(mask_ratio=0.15) # Mask 15% of electrodes randomly

# 2. Initialize Dataset
train_dataset = NeuralPhonemeDataset(
    neural_data=train_dataset_raw['neural_features'],    # Your 8k neural arrays
    phoneme_targets=processed_dataset['seq_class_ids_clean'],  # Your 8k label lists
    transform=transform
)

# 3. Create Loader
train_loader = DataLoader(
    train_dataset,
    batch_size=16,           # Small batch size is better for 8k samples
    shuffle=True,
    collate_fn=collate_fn_brain,
    num_workers=4,           # Parallel data loading
    pin_memory=True          # Faster transfer to GPU
)

In [128]:
def train_one_epoch(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0

    for inputs, targets, input_lens, target_lens in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)

        print(f"Training batch: inputs shape={inputs.shape}, targets shape={targets.shape}")
        
        # log_probs shape: (B, T_patched, Classes)
        log_probs, patched_lens = model(inputs, input_lens)
        
        # IMPORTANT: CTC expects (T, B, C). We transpose B and T.
        # Shape becomes (T_patched, B, Classes)
        log_probs_ctc = log_probs.transpose(0, 1)
        
        loss = criterion(log_probs_ctc, targets, patched_lens, target_lens)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        print(f"Batch Loss: {loss.item():.4f}")
    
    return total_loss / len(dataloader)


In [130]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeuralPatchConformer(patch_size=1).to(device)

# CTC Loss: blank=0 is standard
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

# SOTA Optimizer: AdamW with high weight decay for small 8k datasets
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

# SOTA Scheduler: OneCycleLR handles the "warmup" needed for Transformers
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=5e-4, steps_per_epoch=len(train_loader), epochs=50
)

In [131]:
train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)

/home/rubn/Documents/THU/Projects/BDI-Kaggle_Brain2Text25/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training batch: inputs shape=torch.Size([16, 1335, 512]), targets shape=torch.Size([16, 42])
Batch Loss: 104.4732
Training batch: inputs shape=torch.Size([16, 1060, 512]), targets shape=torch.Size([16, 42])
Batch Loss: 92.9649
Training batch: inputs shape=torch.Size([16, 1498, 512]), targets shape=torch.Size([16, 35])
Batch Loss: 96.4766
Training batch: inputs shape=torch.Size([16, 2209, 512]), targets shape=torch.Size([16, 45])


KeyboardInterrupt: 